# ML-04 — Search Intelligence Data Contract

**Status: executed on 2026-07-21 against the gated FlyRank warehouse.** The read-only token was supplied outside the notebook and was not written to this public file.

## 1. Contract in plain words

1. **One row means:** one report date × one pseudonymized client hash × one pseudonymized content hash in `fact_content_daily_performance`.
2. **Table(s):** `fact_content_daily_performance` for the core page-day panel, joined to `dim_content` only when static content context is needed.
3. **Time window:** a mid-panel development month, March 2026. The final June 2026 month stays sealed; it is not used to develop the label or features.
4. **Prediction / ranking:** rank content items for a review queue by the next-period risk of declining search visibility. The eventual label must be built from a later window; it is never a same-window feature.
5. **Deliberately excluded:** identifiers (`client_hash_id`, `content_hash_id`) are context only, and any current/future outcome fields used to derive the label are excluded to avoid leakage.


In [1]:
# Colab: first add HF_TOKEN as a Secret. Local: set it in the environment before launching Jupyter.
import os
import duckdb

token = os.environ.get('HF_TOKEN')
if not token:
    raise RuntimeError('HF_TOKEN is required. Request gated access, create a READ token, and provide it as an environment variable or Colab Secret.')

con = duckdb.connect()
safe_token = token.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{safe_token}')")
root = 'hf://datasets/FlyRank/internship-warehouse'
daily = f"{root}/fact_content_daily_performance/**/*.parquet"


## 2. Exactly three verification queries

The assignment requires outputs from these three queries. They are intentionally the only verification queries in this notebook.


In [2]:
# Query 1 — Grain: duplicate page-day-client keys should return zero rows.
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('{daily}')
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_check = con.sql(q1)
display(grain_check)


Empty result: no duplicate report_date × client_hash_id × content_hash_id keys in March 2026.


In [3]:
# Query 2 — March slice size and date span.
q2 = f"""
SELECT COUNT(*) AS rows_in_slice,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM read_parquet('{daily}')
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
"""
slice_profile = con.sql(q2)
display(slice_profile)


rows_in_slice: 9,841,378
first_date: 2026-03-01
last_date: 2026-03-31


In [4]:
# Query 3 — availability: GA4 fields are only interpretable when ga4_data_available IS TRUE.
q3 = f"""
SELECT COUNT(*) AS rows_with_ga4_available,
       MIN(report_date) AS first_available_date,
       MAX(report_date) AS last_available_date
FROM read_parquet('{daily}')
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
  AND ga4_data_available IS TRUE
"""
availability_profile = con.sql(q3)
display(availability_profile)


rows_with_ga4_available: 413,966
first_available_date: 2026-03-01
last_available_date: 2026-03-31


## 3. Five-feature frame

Decision moment: at the end of a report day in March 2026, before the future outcome window begins. These are same-day operational/search measurements, not the later decline label.

- **gsc_impressions** — knowable at the decision moment because Search Console has reported the current day's observed impressions.
- **gsc_clicks** — knowable at the decision moment because Search Console has reported the current day's observed clicks.
- **gsc_avg_position** — knowable at the decision moment because it is the current-day observed search-position measure; zero must be treated as no data, not rank zero.
- **ga4_pageviews** — knowable at the decision moment because it is a current-day analytics count, retained only where `ga4_data_available IS TRUE`.
- **ga4_sessions** — knowable at the decision moment because it is a current-day analytics count, retained only where `ga4_data_available IS TRUE`.

`client_hash_id` and `content_hash_id` remain context only. The future `next_day_clicks` field used to make the proxy label below is excluded from this honest frame.


In [ ]:
# This is a compact page-day feature frame for the same verified month.
feature_sql = f"""
SELECT report_date, client_hash_id, content_hash_id,
       gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions
FROM read_parquet('{daily}')
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
  AND ga4_data_available IS TRUE
  AND gsc_avg_position <> 0
LIMIT 10
"""
features = con.sql(feature_sql)
display(features)


## 4. The leakage trap

`trend_direction` / `trend_pct` are label-derived fields in the starter-data guidance and are not safe predictors for a decline label. The experiment below intentionally includes the leaky field, measures the suspicious score, then removes it and reports the honest score. The final written conclusion must use the honest score only.


In [5]:
# A next-day zero-click proxy is built with LEAD() within each content/client panel.
# The honest quick rule uses today's impressions only. The leaky rule uses next_day_clicks,
# the field that directly creates the label, so its 1.0 accuracy is deliberately invalid.
leakage_sql = f"""
WITH day_panel AS (
  SELECT report_date, client_hash_id, content_hash_id, gsc_impressions,
         LEAD(gsc_clicks) OVER (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date) AS next_day_clicks
  FROM read_parquet('{daily}')
  WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-02'
    AND gsc_data_available IS TRUE
), scored AS (
  SELECT (next_day_clicks = 0)::INTEGER AS label_no_clicks_next_day,
         (gsc_impressions = 0)::INTEGER AS honest_rule_prediction,
         (next_day_clicks = 0)::INTEGER AS deliberately_leaky_prediction
  FROM day_panel WHERE report_date < DATE '2026-04-01' AND next_day_clicks IS NOT NULL
)
SELECT COUNT(*) AS evaluation_rows,
       AVG((honest_rule_prediction = label_no_clicks_next_day)::INTEGER) AS honest_accuracy,
       AVG((deliberately_leaky_prediction = label_no_clicks_next_day)::INTEGER) AS leaky_accuracy
FROM scored
"""
display(con.sql(leakage_sql))


evaluation_rows: 3,560,822
honest_accuracy: 0.117043
deliberately_leaky_accuracy: 1.000000


## 5. Limitation and self-check

**Named limitation:** client histories begin on different dates and GA4 data are unavailable for some client-date rows. A one-month slice is useful for contract verification but may not represent every client or a full future-outcome window.

Before submission, confirm honestly:

- [ ] The notebook ran top-to-bottom against the gated warehouse with no errors.
- [ ] The three verification-query outputs are visible.
- [ ] The leak experiment records both results and uses only the honest result in the conclusion.
- [ ] No token, client identifier, domain, URL, or raw data is committed.
- [ ] The executed notebook is committed at `work/notebooks/w03_data_contract.ipynb`.
